# Demo - Time Travel and Zero-Copy Clone (about 3 minutes)

**Why we are doing this.** Three questions every data engineer asks:
1. *Someone ran a bad update. Can I undo it?*
2. *Someone dropped a table. Can I get it back?*
3. *Where do I test new code against real data?*

| Feature | What it does | Today in Synapse / Azure SQL |
|---|---|---|
| **Time Travel** | Query or restore a table as it was at any point in the retention window | Restore a backup to a new database, then copy rows back |
| **UNDROP** | Bring back a dropped table, schema or database | Restore from backup |
| **Zero-copy clone** | A full copy of a table in seconds, no extra storage until it changes | Copy the data, pay for it twice, wait |

We use one standalone table, `GOLD.CLAIMS_SNAPSHOT`, a copy of the published claims. Nothing else depends on it.

### RUN DURING DEMO
Set the role, database and warehouse. Run this first.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_BI;
USE SCHEMA GOLD;

### SETUP - run before the demo
Create the demo table from the published claims (about 600K rows). Needs 03 Part B to have run.

In [ ]:
%%sql -r setup
CREATE OR REPLACE TABLE GOLD.CLAIMS_SNAPSHOT AS
SELECT CLAIM_ID, MEMBER_ID, CLIENT_ID, SERVICE_DATE, CPT_CODE, PAID_AMOUNT, CLAIM_STATUS
FROM GOLD.CLAIMS_PUBLISHED;

## Part 1 - Undo a bad update

### RUN DURING DEMO
Totals before anything goes wrong.

In [ ]:
%%sql -r totals_start
SELECT CLIENT_ID, TO_CHAR(SUM(PAID_AMOUNT), '$999,999,999') AS TOTAL_PAID
FROM GOLD.CLAIMS_SNAPSHOT
GROUP BY CLIENT_ID ORDER BY CLIENT_ID;

### RUN DURING DEMO
**The bad update.** The plan was to zero out *denied* Aetna claims. The `CLAIM_STATUS` filter was forgotten, so every Aetna claim is zeroed. We note the time first, so we can go back to it.

In [ ]:
%%sql -r bad_update
SET BEFORE_UPDATE = CURRENT_TIMESTAMP();

UPDATE GOLD.CLAIMS_SNAPSHOT
SET PAID_AMOUNT = 0
WHERE CLIENT_ID = 'AETNA_DEMO';   -- forgot: AND CLAIM_STATUS = 'DENIED'

### RUN DURING DEMO
The table now. Aetna shows $0.

In [ ]:
%%sql -r totals_now
SELECT CLIENT_ID, TO_CHAR(SUM(PAID_AMOUNT), '$999,999,999') AS TOTAL_PAID
FROM GOLD.CLAIMS_SNAPSHOT
GROUP BY CLIENT_ID ORDER BY CLIENT_ID;

### RUN DURING DEMO
The same table, as it was **before the update**. Just add `AT(...)` to the query.

In [ ]:
%%sql -r totals_before
SELECT CLIENT_ID, TO_CHAR(SUM(PAID_AMOUNT), '$999,999,999') AS TOTAL_PAID
FROM GOLD.CLAIMS_SNAPSHOT AT(TIMESTAMP => $BEFORE_UPDATE)
GROUP BY CLIENT_ID ORDER BY CLIENT_ID;

### RUN DURING DEMO
Undo: copy the old amounts back from the past version, then check.

In [ ]:
%%sql -r undo_update
UPDATE GOLD.CLAIMS_SNAPSHOT t
SET PAID_AMOUNT = o.PAID_AMOUNT
FROM GOLD.CLAIMS_SNAPSHOT AT(TIMESTAMP => $BEFORE_UPDATE) o
WHERE t.CLAIM_ID = o.CLAIM_ID
  AND t.CLIENT_ID = 'AETNA_DEMO';

SELECT CLIENT_ID, TO_CHAR(SUM(PAID_AMOUNT), '$999,999,999') AS TOTAL_PAID
FROM GOLD.CLAIMS_SNAPSHOT
GROUP BY CLIENT_ID ORDER BY CLIENT_ID;

## Part 2 - Undrop a table

### RUN DURING DEMO
**Oops.** The table is dropped by mistake.

In [ ]:
%%sql -r drop_table
DROP TABLE GOLD.CLAIMS_SNAPSHOT;

### RUN DURING DEMO
It's gone: no rows returned.

In [ ]:
%%sql -r is_gone
SHOW TABLES LIKE 'CLAIMS_SNAPSHOT' IN SCHEMA SOMATUS_DEMO.GOLD;

### RUN DURING DEMO
One command brings it back, with all its data.

In [ ]:
%%sql -r undrop
UNDROP TABLE GOLD.CLAIMS_SNAPSHOT;

SELECT COUNT(*) AS CLAIMS FROM GOLD.CLAIMS_SNAPSHOT;

## Part 3 - Zero-copy clone

### RUN DURING DEMO
Copy the gold table for testing new code. It takes seconds and uses no extra storage until something changes.

In [ ]:
%%sql -r clone
CREATE OR REPLACE TABLE GOLD.CLAIMS_SNAPSHOT_DEV CLONE GOLD.CLAIMS_SNAPSHOT;

### RUN DURING DEMO
Same rows in both.

In [ ]:
%%sql -r clone_check
SELECT 'PROD' AS COPY, COUNT(*) AS CLAIMS FROM GOLD.CLAIMS_SNAPSHOT UNION ALL
SELECT 'DEV',  COUNT(*) FROM GOLD.CLAIMS_SNAPSHOT_DEV;

## What we just proved
1. **Bad update:** seen and undone with one line of SQL. No backup, no ticket.
2. **Dropped table:** back with one command, data and all.
3. **Zero-copy clone:** a full copy of a gold table in seconds, for testing new code against real data before production.

**IF ASKED / FALLBACK:** if `$BEFORE_UPDATE` is not set (for example, the notebook reconnected), use `AT(OFFSET => -120)` instead, meaning two minutes ago.